# Suite2 Project9 Association

**Dataset:** Online Retail (UCI, 5,000 transactions)

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 9: Market Basket — Association Rules"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('online_retail.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite2_project9_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 9: MARKET BASKET ANALYSIS — ASSOCIATION RULES

In [ ]:
print(f"Dataset: Online Retail ({df.shape[0]:,} rows)")
print(f"Invoices: {df['InvoiceNo'].nunique():,}")
print(f"Products: {df['Description'].nunique()}")
print(f"\nExample transactions:")
print(df.head(10).to_string())

# ── 1. Transaction Encoding ──

## 1. Preparing Transaction Data

In [ ]:
# Group items by invoice
transactions = df.groupby('InvoiceNo')['Description'].apply(list)
print(f"Number of baskets: {len(transactions):,}")

# Stats
basket_sizes = transactions.apply(len)
print(f"\nBasket size stats:")
print(f"  Mean:     {basket_sizes.mean():.1f} items")
print(f"  Median:   {basket_sizes.median():.0f} items")
print(f"  Min/Max:  {basket_sizes.min()} / {basket_sizes.max()}")
print(f"  >20 items: {(basket_sizes > 20).sum()} baskets")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(basket_sizes, bins=30, color='#2e86de', edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Items per Basket', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Basket Sizes', fontsize=13)
axes[0].axvline(x=basket_sizes.mean(), color='red', linestyle='--', label=f'Mean={basket_sizes.mean():.1f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top products
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(20)
axes[1].barh(range(len(top_products)), top_products.values, color='#2e86de', alpha=0.7)
axes[1].set_yticks(range(len(top_products)))
axes[1].set_yticklabels(top_products.index, fontsize=9)
axes[1].set_xlabel('Total Quantity Sold', fontsize=12)
axes[1].set_title('Top 20 Products by Quantity', fontsize=13)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')
plt.tight_layout()
# 'p9_baskets_products')
plt.show()
print("[Saved] p9_baskets_products.png")

# ── 2. Apriori Algorithm (manual implementation) ──

## 2. Association Rules — Apriori (Simplified)

In [ ]:
# Get frequent items for analysis
min_support = 0.02
all_items = [item for basket in transactions for item in basket]
item_counts = pd.Series(all_items).value_counts()
n_baskets = len(transactions)
frequent_items = item_counts[item_counts / n_baskets >= min_support]
print(f"Total unique items: {item_counts.shape[0]}")
print(f"Frequent items (support ≥ {min_support}): {frequent_items.shape[0]}")
print(f"\nTop 10 frequent items:")
for item, count in frequent_items.head(10).items():
    support = count / n_baskets
    print(f"  {item:45s}: count={count:5d}, support={support:.4f}")

# Calculate pairwise co-occurrence for top frequent items

## 3. Pair Co-occurrence & Lift Analysis

In [ ]:
top_items = list(frequent_items.head(10).index)
co_occurrence = pd.DataFrame(0, index=top_items, columns=top_items, dtype=float)

for basket in transactions:
    items_in_basket = [item for item in basket if item in top_items]
    for i in range(len(items_in_basket)):
        for j in range(i+1, len(items_in_basket)):
            co_occurrence.loc[items_in_basket[i], items_in_basket[j]] += 1
            co_occurrence.loc[items_in_basket[j], items_in_basket[i]] += 1

# Calculate lift
rules = []
for item_a in top_items:
    for item_b in top_items:
        if item_a >= item_b:  # avoid duplicates
            continue
        count_a = item_counts[item_a]
        count_b = item_counts[item_b]
        count_ab = co_occurrence.loc[item_a, item_b]
        
        support_a = count_a / n_baskets
        support_b = count_b / n_baskets
        support_ab = count_ab / n_baskets
        
        if support_ab > 0:
            confidence = support_ab / support_a
            lift = support_ab / (support_a * support_b)
            if lift > 1.0:  # positive correlation
                rules.append({
                    'Item A': item_a[:30],
                    'Item B': item_b[:30],
                    'Support': f'{support_ab:.4f}',
                    'Confidence': f'{confidence:.3f}',
                    'Lift': f'{lift:.2f}'
                })

rules_df = pd.DataFrame(rules)
if len(rules_df) > 0:
    rules_df = rules_df.sort_values('Lift', ascending=False)
    print("Top 10 Association Rules (by Lift):")
    print(rules_df.head(10).to_string(index=False))

# Visualize co-occurrence network
fig, ax = plt.subplots(figsize=(12, 10))
# Simple co-occurrence heatmap
sns.heatmap(co_occurrence, annot=True, fmt='.0f', cmap='YlOrRd', 
            linewidths=1, ax=ax, cbar_kws={'label': 'Co-occurrence Count'})
ax.set_title('Item Co-occurrence Matrix (Top 10 Frequent Items)', fontsize=14)
plt.tight_layout()
# 'p9_cooccurrence_matrix')
plt.show()
print("[Saved] p9_cooccurrence_matrix.png")

results = {
    'n_transactions': int(len(transactions)),
    'n_unique_products': int(item_counts.shape[0]),
    'frequent_items_count': int(frequent_items.shape[0]),
    'min_support': min_support,
    'top_frequent': frequent_items.head(10).to_dict(),
    'top_rules': rules_df.head(10).to_dict('records') if len(rules_df) > 0 else [],
    'avg_basket_size': float(basket_sizes.mean()),
}
    json.dump(results, f, indent=2, default=str)

print("Done.")